# Engage Corpus: Neural Collaborative Filtering Training

This notebook trains **TWO separate neural collaborative filtering (NCF) models** for subreddit recommendation, following the Engage Corpus paper (Cheng et al., 2022):

## The Two Models

### 1. **NCF Baseline** (Pure Collaborative Filtering)
- Uses ONLY user-item interaction patterns
- No text features or context
- GMF + MLP fusion architecture (He et al., 2017)
- Can be trained with data from `process_ncf_baseline.py`

### 2. **NCF with Context** (Hybrid: Collaborative + Content)
- Uses user-item interactions + VW text-based context
- VW generates 5000-dim probability scores from user text
- Context is projected to 128-dim and concatenated with user embeddings

## Architecture Details

### NCF Baseline (Model 1)
```
User_emb ⊙ Item_emb → GMF pathway
User_emb ⊕ Item_emb → MLP pathway → MLP output
Concat(GMF, MLP) → Final layer → Sigmoid
```

### NCF with Context (Model 2)
```
VW scores (5000-dim) → Linear projection (128-dim) → Context_proj
User_emb ⊕ Context_proj → Enhanced_user_emb
Enhanced_user_emb ⊙ Item_emb → GMF pathway
Enhanced_user_emb ⊕ Item_emb → MLP pathway → MLP output
Concat(GMF, MLP) → Final layer → Sigmoid
```

## Setup and Data Loading

In [ ]:
# Install required packages
!pip install torch numpy tqdm scikit-learn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import pickle
from tqdm import tqdm
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

In [ ]:
# Set paths to your data in Google Drive
DATA_DIR = '/content/drive/MyDrive/engage_corpus_processed'
NCF_DIR = os.path.join(DATA_DIR, 'ncf_data')

# Verify paths exist
assert os.path.exists(DATA_DIR), f"Data directory not found: {DATA_DIR}"
assert os.path.exists(NCF_DIR), f"NCF data directory not found: {NCF_DIR}"

print("Data directories found!")
print(f"Contents of {DATA_DIR}:")
print(os.listdir(DATA_DIR))
print(f"\nContents of {NCF_DIR}:")
print(os.listdir(NCF_DIR))

In [ ]:
# Load metadata
with open(os.path.join(DATA_DIR, 'metadata.json'), 'r') as f:
    metadata = json.load(f)

with open(os.path.join(DATA_DIR, 'user_mapping.json'), 'r') as f:
    user_mapping = json.load(f)

NUM_USERS = user_mapping['num_users']
NUM_SUBREDDITS = metadata['num_subreddits']

print(f"Number of users: {NUM_USERS}")
print(f"Number of subreddits: {NUM_SUBREDDITS}")

## Data Loading Functions

In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_interactions(filepath):
    """
    Load user-subreddit interaction pairs.
    
    Returns:
        users: np.array of user IDs
        subreddits: np.array of subreddit IDs
    """
    users = []
    subreddits = []
    
    with open(filepath, 'r') as f:
        for line in f:
            user_id, sub_id = line.strip().split('\t')
            users.append(int(user_id))
            subreddits.append(int(sub_id))
    
    return np.array(users), np.array(subreddits)

# Load the Single Static Context Map (for context model)
print("Loading static user context map...")
user_context_map_path = os.path.join(NCF_DIR, 'user_context_map.npy')

if os.path.exists(user_context_map_path):
    user_context_map = np.load(user_context_map_path)
    print(f"Loaded user context map: {user_context_map.shape}")
else:
    print("Context map not found - you can only train the baseline model")
    print("  Run process_engage_corpus_v3.py to generate context for the context model")
    user_context_map = None

# Load Interactions for all splits
print("\nLoading interactions...")
train_users, train_subreddits = load_interactions(os.path.join(NCF_DIR, 'train.tsv'))
dev_users, dev_subreddits = load_interactions(os.path.join(NCF_DIR, 'dev.tsv'))
test_users, test_subreddits = load_interactions(os.path.join(NCF_DIR, 'test.tsv'))

# Print Statistics
print(f"Train samples: {len(train_users)}")
print(f"Dev samples:   {len(dev_users)}")
print(f"Test samples:  {len(test_users)}")
if user_context_map is not None:
    print(f"Context map shape: {user_context_map.shape}")
    print(f"Context non-zero ratio: {(user_context_map > 0).sum() / user_context_map.size:.4f}")

## Dataset Class

In [ ]:
# ============================================================================
# DATASET CLASSES (SEPARATE FOR BASELINE AND CONTEXT MODELS)
# ============================================================================

class NCFDatasetBaseline(Dataset):
    """
    NCF Dataset for BASELINE model (NO CONTEXT).
    
    Pure collaborative filtering based only on user-item interactions.
    """
    
    def __init__(self, users, subreddits, num_subreddits, 
                 num_negatives=4, is_training=True):
        """
        Args:
            users: Array of user IDs
            subreddits: Array of subreddit IDs (positive samples)
            num_subreddits: Total number of subreddits
            num_negatives: Number of negative samples per positive
            is_training: Whether this is training data
        """
        self.users = users
        self.subreddits = subreddits
        self.num_subreddits = num_subreddits
        self.num_negatives = num_negatives
        self.is_training = is_training
        
        # Build user interaction sets for negative sampling
        if is_training:
            self.user_interactions = {}
            for u, s in zip(users, subreddits):
                if u not in self.user_interactions:
                    self.user_interactions[u] = set()
                self.user_interactions[u].add(s)
    
    def __len__(self):
        if self.is_training:
            return len(self.users) * (1 + self.num_negatives)
        else:
            return len(self.users)
    
    def __getitem__(self, idx):
        if self.is_training:
            # Training: return positive and negative samples
            sample_idx = idx // (1 + self.num_negatives)
            sample_type = idx % (1 + self.num_negatives)
            
            user = self.users[sample_idx]
            
            if sample_type == 0:
                # Positive sample
                subreddit = self.subreddits[sample_idx]
                label = 1.0
            else:
                # Negative sample
                subreddit = np.random.randint(0, self.num_subreddits)
                while subreddit in self.user_interactions.get(user, set()):
                    subreddit = np.random.randint(0, self.num_subreddits)
                label = 0.0
            
            return {
                'user': torch.tensor(user, dtype=torch.long),
                'subreddit': torch.tensor(subreddit, dtype=torch.long),
                'label': torch.tensor(label, dtype=torch.float32)
            }
        else:
            # Evaluation: return only positive samples
            user = self.users[idx]
            
            return {
                'user': torch.tensor(user, dtype=torch.long),
                'subreddit': torch.tensor(self.subreddits[idx], dtype=torch.long),
                'label': torch.tensor(1.0, dtype=torch.float32)
            }


class NCFDatasetWithContext(Dataset):
    """
    NCF Dataset with VW context (for context-aware model).
    
    Uses static context map - ONE 5000-dim vector per user.
    """
    
    def __init__(self, users, subreddits, user_context_map, num_subreddits, 
                 num_negatives=4, is_training=True):
        """
        Args:
            users: Array of user IDs
            subreddits: Array of subreddit IDs (positive samples)
            user_context_map: Array of shape (num_users, 5000) with context vectors
            num_subreddits: Total number of subreddits
            num_negatives: Number of negative samples per positive
            is_training: Whether this is training data
        """
        self.users = users
        self.subreddits = subreddits
        self.user_context_map = user_context_map
        self.num_subreddits = num_subreddits
        self.num_negatives = num_negatives
        self.is_training = is_training
        
        # Build user interaction sets for negative sampling
        if is_training:
            self.user_interactions = {}
            for u, s in zip(users, subreddits):
                if u not in self.user_interactions:
                    self.user_interactions[u] = set()
                self.user_interactions[u].add(s)
    
    def __len__(self):
        if self.is_training:
            return len(self.users) * (1 + self.num_negatives)
        else:
            return len(self.users)
    
    def __getitem__(self, idx):
        if self.is_training:
            # Training: return positive and negative samples
            sample_idx = idx // (1 + self.num_negatives)
            sample_type = idx % (1 + self.num_negatives)
            
            user = self.users[sample_idx]
            context = self.user_context_map[user]
            
            if sample_type == 0:
                # Positive sample
                subreddit = self.subreddits[sample_idx]
                label = 1.0
            else:
                # Negative sample
                subreddit = np.random.randint(0, self.num_subreddits)
                while subreddit in self.user_interactions.get(user, set()):
                    subreddit = np.random.randint(0, self.num_subreddits)
                label = 0.0
            
            return {
                'user': torch.tensor(user, dtype=torch.long),
                'subreddit': torch.tensor(subreddit, dtype=torch.long),
                'context': torch.tensor(context, dtype=torch.float32),
                'label': torch.tensor(label, dtype=torch.float32)
            }
        else:
            # Evaluation: return only positive samples
            user = self.users[idx]
            context = self.user_context_map[user]
            
            return {
                'user': torch.tensor(user, dtype=torch.long),
                'subreddit': torch.tensor(self.subreddits[idx], dtype=torch.long),
                'context': torch.tensor(context, dtype=torch.float32),
                'label': torch.tensor(1.0, dtype=torch.float32)
            }

## Model Definitions

Following the paper's architecture (Figure 1).

In [ ]:
class NCF(nn.Module):
    """Neural Collaborative Filtering baseline model.
    
    Architecture from He et al. (2017):
    - GMF pathway: Element-wise product of user and item embeddings
    - MLP pathway: Concatenation of embeddings through MLP layers
    - Fusion: Concatenate GMF and MLP outputs, then final prediction layer
    """
    
    def __init__(self, num_users, num_subreddits, embedding_dim=64, hidden_layers=[128, 64, 32]):
        super(NCF, self).__init__()
        
        # GMF Embeddings
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.subreddit_embedding_gmf = nn.Embedding(num_subreddits, embedding_dim)
        
        # MLP Embeddings (separate from GMF)
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.subreddit_embedding_mlp = nn.Embedding(num_subreddits, embedding_dim)
        
        # MLP layers
        mlp_layers = []
        input_dim = embedding_dim * 2
        
        for hidden_dim in hidden_layers:
            mlp_layers.append(nn.Linear(input_dim, hidden_dim))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            input_dim = hidden_dim
        
        self.mlp = nn.Sequential(*mlp_layers)
        
        # Final prediction layer (combines GMF and MLP)
        # Input: embedding_dim (from GMF) + hidden_layers[-1] (from MLP)
        self.final_layer = nn.Linear(embedding_dim + hidden_layers[-1], 1)
        
        # Initialize embeddings
        nn.init.normal_(self.user_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.subreddit_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.user_embedding_mlp.weight, std=0.01)
        nn.init.normal_(self.subreddit_embedding_mlp.weight, std=0.01)
    
    def forward(self, user, subreddit):
        # GMF pathway: element-wise product
        user_emb_gmf = self.user_embedding_gmf(user)
        sub_emb_gmf = self.subreddit_embedding_gmf(subreddit)
        gmf_output = user_emb_gmf * sub_emb_gmf  # Element-wise product
        
        # MLP pathway: concatenation through MLP
        user_emb_mlp = self.user_embedding_mlp(user)
        sub_emb_mlp = self.subreddit_embedding_mlp(subreddit)
        mlp_input = torch.cat([user_emb_mlp, sub_emb_mlp], dim=-1)
        mlp_output = self.mlp(mlp_input)
        
        # Fusion: concatenate GMF and MLP outputs
        fusion = torch.cat([gmf_output, mlp_output], dim=-1)
        
        # Final prediction
        output = torch.sigmoid(self.final_layer(fusion))
        
        return output.squeeze()


class NCFWithContext(nn.Module):
    """Neural Collaborative Filtering with VW context scores.
    
    Architecture from Engage Corpus paper (Figure 1):
    - Context processing: 5000-dim → Linear(128-dim)
    - User enhancement: Concat(User_emb, Context_proj)
    - GMF pathway: Enhanced_user_emb ⊙ Item_emb
    - MLP pathway: Concat(Enhanced_user_emb, Item_emb) → MLP
    - Fusion: Concat(GMF, MLP) → Final layer
    """
    
    def __init__(self, num_users, num_subreddits, embedding_dim=64, context_dim=5000, 
                 context_projection_dim=128, hidden_layers=[128, 64, 32]):
        super(NCFWithContext, self).__init__()
        
        # Context projection (5000 → 128)
        self.context_projection = nn.Linear(context_dim, context_projection_dim)
        
        # Enhanced user embedding size (original + projected context)
        enhanced_user_dim = embedding_dim + context_projection_dim
        
        # GMF Embeddings
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.subreddit_embedding_gmf = nn.Embedding(num_subreddits, enhanced_user_dim)
        
        # MLP Embeddings (separate from GMF)
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.subreddit_embedding_mlp = nn.Embedding(num_subreddits, embedding_dim)
        
        # MLP layers (input is enhanced_user + subreddit)
        mlp_layers = []
        input_dim = enhanced_user_dim + embedding_dim
        
        for hidden_dim in hidden_layers:
            mlp_layers.append(nn.Linear(input_dim, hidden_dim))
            mlp_layers.append(nn.ReLU())
            mlp_layers.append(nn.Dropout(0.2))
            input_dim = hidden_dim
        
        self.mlp = nn.Sequential(*mlp_layers)
        
        # Final prediction layer
        self.final_layer = nn.Linear(enhanced_user_dim + hidden_layers[-1], 1)
        
        # Initialize embeddings
        nn.init.normal_(self.user_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.subreddit_embedding_gmf.weight, std=0.01)
        nn.init.normal_(self.user_embedding_mlp.weight, std=0.01)
        nn.init.normal_(self.subreddit_embedding_mlp.weight, std=0.01)
    
    def forward(self, user, subreddit, context):
        # Project context to 128-dim
        context_proj = self.context_projection(context)
        
        # Create enhanced user embedding (original + context)
        user_emb_gmf = self.user_embedding_gmf(user)
        enhanced_user_gmf = torch.cat([user_emb_gmf, context_proj], dim=-1)
        
        user_emb_mlp = self.user_embedding_mlp(user)
        enhanced_user_mlp = torch.cat([user_emb_mlp, context_proj], dim=-1)
        
        # GMF pathway: element-wise product
        sub_emb_gmf = self.subreddit_embedding_gmf(subreddit)
        gmf_output = enhanced_user_gmf * sub_emb_gmf
        
        # MLP pathway: concatenation through MLP
        sub_emb_mlp = self.subreddit_embedding_mlp(subreddit)
        mlp_input = torch.cat([enhanced_user_mlp, sub_emb_mlp], dim=-1)
        mlp_output = self.mlp(mlp_input)
        
        # Fusion: concatenate GMF and MLP outputs
        fusion = torch.cat([gmf_output, mlp_output], dim=-1)
        
        # Final prediction
        output = torch.sigmoid(self.final_layer(fusion))
        
        return output.squeeze()

## Training Function

In [ ]:
def train_epoch_baseline(model, dataloader, criterion, optimizer, device):
    """Train baseline model for one epoch (no context)."""
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc="Training"):
        user = batch['user'].to(device)
        subreddit = batch['subreddit'].to(device)
        label = batch['label'].to(device)
        
        optimizer.zero_grad()
        output = model(user, subreddit)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


def validate_epoch_baseline(model, dataloader, criterion, device):
    """Validate baseline model for one epoch (no context)."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            user = batch['user'].to(device)
            subreddit = batch['subreddit'].to(device)
            label = batch['label'].to(device)
            
            output = model(user, subreddit)
            loss = criterion(output, label)
            
            total_loss += loss.item()
    
    return total_loss / len(dataloader)


def train_epoch_context(model, dataloader, criterion, optimizer, device):
    """Train context model for one epoch (with context)."""
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc="Training"):
        user = batch['user'].to(device)
        subreddit = batch['subreddit'].to(device)
        context = batch['context'].to(device)
        label = batch['label'].to(device)
        
        optimizer.zero_grad()
        output = model(user, subreddit, context)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


def validate_epoch_context(model, dataloader, criterion, device):
    """Validate context model for one epoch (with context)."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            user = batch['user'].to(device)
            subreddit = batch['subreddit'].to(device)
            context = batch['context'].to(device)
            label = batch['label'].to(device)
            
            output = model(user, subreddit, context)
            loss = criterion(output, label)
            
            total_loss += loss.item()
    
    return total_loss / len(dataloader)


def predict_all_subreddits_baseline(model, user_id, num_subreddits, device, batch_size=512):
    """Predict scores for all subreddits for a given user (baseline model)."""
    model.eval()
    scores = []
    
    with torch.no_grad():
        for start_idx in range(0, num_subreddits, batch_size):
            end_idx = min(start_idx + batch_size, num_subreddits)
            batch_size_actual = end_idx - start_idx
            
            user_batch = torch.tensor([user_id] * batch_size_actual, dtype=torch.long).to(device)
            sub_batch = torch.arange(start_idx, end_idx, dtype=torch.long).to(device)
            
            batch_scores = model(user_batch, sub_batch)
            scores.append(batch_scores.cpu().numpy())
    
    return np.concatenate(scores)


def predict_all_subreddits_context(model, user_id, context, num_subreddits, device, batch_size=512):
    """Predict scores for all subreddits for a given user (context model)."""
    model.eval()
    scores = []
    
    with torch.no_grad():
        for start_idx in range(0, num_subreddits, batch_size):
            end_idx = min(start_idx + batch_size, num_subreddits)
            batch_size_actual = end_idx - start_idx
            
            user_batch = torch.tensor([user_id] * batch_size_actual, dtype=torch.long).to(device)
            sub_batch = torch.arange(start_idx, end_idx, dtype=torch.long).to(device)
            context_batch = context.unsqueeze(0).repeat(batch_size_actual, 1).to(device)
            
            batch_scores = model(user_batch, sub_batch, context_batch)
            scores.append(batch_scores.cpu().numpy())
    
    return np.concatenate(scores)

## Train NCF Baseline (GMF + MLP)

In [ ]:
print("="*60)
print("MODEL 1: NCF BASELINE (GMF + MLP, NO CONTEXT)")
print("="*60)
print("This model uses ONLY collaborative filtering (user-item interactions)")
print("No text features or VW context are used.")
print()

# Hyperparameters
EMBEDDING_DIM = 64
BATCH_SIZE = 256
LEARNING_RATE = 0.0005
NUM_EPOCHS = 15
NUM_NEGATIVES = 4
CHECKPOINT_INTERVAL = 3  # Save checkpoint every 3 epochs

# Checkpoint directory
CHECKPOINT_DIR = '/content/drive/MyDrive/ncf_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Create dataset and dataloader FOR BASELINE (no context)
train_dataset_baseline = NCFDatasetBaseline(train_users, train_subreddits, NUM_SUBREDDITS, 
                                          num_negatives=NUM_NEGATIVES, is_training=True)
train_loader_baseline = DataLoader(train_dataset_baseline, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Create VALIDATION dataset and dataloader
dev_dataset_baseline = NCFDatasetBaseline(dev_users, dev_subreddits, NUM_SUBREDDITS,
                                         num_negatives=NUM_NEGATIVES, is_training=True)
dev_loader_baseline = DataLoader(dev_dataset_baseline, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Create model
ncf_baseline = NCF(NUM_USERS, NUM_SUBREDDITS, embedding_dim=EMBEDDING_DIM).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(ncf_baseline.parameters(), lr=LEARNING_RATE)

# Check for existing checkpoint
start_epoch = 0
baseline_checkpoint_pattern = os.path.join(CHECKPOINT_DIR, 'ncf_baseline_epoch_*.pth')
import glob
existing_checkpoints = sorted(glob.glob(baseline_checkpoint_pattern))
if existing_checkpoints:
    latest_checkpoint = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest_checkpoint}")
    checkpoint = torch.load(latest_checkpoint)
    ncf_baseline.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {start_epoch}")

print(f"\nModel parameters: {sum(p.numel() for p in ncf_baseline.parameters()):,}")
print(f"Training samples: {len(train_dataset_baseline):,}")
print(f"Validation samples: {len(dev_dataset_baseline):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per epoch: {len(train_loader_baseline)}")
print()

# Training loop
for epoch in range(start_epoch, NUM_EPOCHS):
    train_loss = train_epoch_baseline(ncf_baseline, train_loader_baseline, criterion, optimizer, device)
    val_loss = validate_epoch_baseline(ncf_baseline, dev_loader_baseline, criterion, device)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    # Save checkpoint every CHECKPOINT_INTERVAL epochs
    if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
        checkpoint_path = os.path.join(CHECKPOINT_DIR, f'ncf_baseline_epoch_{epoch+1}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': ncf_baseline.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss
        }, checkpoint_path)
        print(f" Checkpoint saved: {checkpoint_path}")

# Save final model
final_model_path = os.path.join(CHECKPOINT_DIR, 'ncf_baseline_final.pth')
torch.save(ncf_baseline.state_dict(), final_model_path)
print(f"\nNCF Baseline final model saved to {final_model_path}")

## Generate Predictions from NCF Baseline

In [ ]:
print("="*60)
print("GENERATING PREDICTIONS FROM NCF BASELINE")
print("="*60)

# ============================================================
# SPECIFY WHICH CHECKPOINT TO USE FOR PREDICTIONS
# ============================================================
# Options:
# 1. Use 'ncf_baseline_final.pth' for the final trained model
# 2. Use 'ncf_baseline_epoch_15.pth' for a specific epoch
# 3. Use 'ncf_baseline_epoch_12.pth', etc.

BASELINE_CHECKPOINT_NAME = 'ncf_baseline_final.pth'  # Change this to use different checkpoint
baseline_checkpoint_path = os.path.join(CHECKPOINT_DIR, BASELINE_CHECKPOINT_NAME)

# Verify checkpoint exists
if not os.path.exists(baseline_checkpoint_path):
    print(f"ERROR: Checkpoint not found: {baseline_checkpoint_path}")
    print(f"\nAvailable baseline checkpoints:")
    baseline_checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'ncf_baseline_*.pth')))
    for ckpt in baseline_checkpoints:
        print(f"  - {os.path.basename(ckpt)}")
    raise FileNotFoundError(f"Checkpoint not found: {baseline_checkpoint_path}")

print(f"Loading weights from: {BASELINE_CHECKPOINT_NAME}")

# Create a fresh model instance
ncf_baseline_eval = NCF(NUM_USERS, NUM_SUBREDDITS, embedding_dim=EMBEDDING_DIM).to(device)

# Load weights from checkpoint
checkpoint_data = torch.load(baseline_checkpoint_path, map_location=device)

# Handle both formats: full checkpoint dict or just state_dict
if isinstance(checkpoint_data, dict) and 'model_state_dict' in checkpoint_data:
    ncf_baseline_eval.load_state_dict(checkpoint_data['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint_data.get('epoch', 'final')}")
    if 'val_loss' in checkpoint_data:
        print(f"  Validation loss: {checkpoint_data['val_loss']:.4f}")
else:
    # Assume it's just the state_dict
    ncf_baseline_eval.load_state_dict(checkpoint_data)
    print(f"Loaded model weights")

ncf_baseline_eval.eval()

# Dev predictions
print("\nGenerating dev predictions...")
dev_predictions_baseline = np.zeros((len(dev_users), NUM_SUBREDDITS), dtype=np.float32)
for i in tqdm(range(len(dev_users)), desc="Dev predictions"):
    user_id = dev_users[i]
    scores = predict_all_subreddits_baseline(ncf_baseline_eval, user_id, NUM_SUBREDDITS, device)
    dev_predictions_baseline[i] = scores

# Test predictions
print("Generating test predictions...")
test_predictions_baseline = np.zeros((len(test_users), NUM_SUBREDDITS), dtype=np.float32)
for i in tqdm(range(len(test_users)), desc="Test predictions"):
    user_id = test_users[i]
    scores = predict_all_subreddits_baseline(ncf_baseline_eval, user_id, NUM_SUBREDDITS, device)
    test_predictions_baseline[i] = scores

# Save predictions as numpy arrays (for evaluate.py)
np.save('ncf_baseline_dev_predictions.npy', dev_predictions_baseline)
np.save('ncf_baseline_test_predictions.npy', test_predictions_baseline)
np.save('ncf_baseline_dev_ground_truth.npy', dev_subreddits)
np.save('ncf_baseline_test_ground_truth.npy', test_subreddits)

print("\nPredictions saved!")
print(f"  - ncf_baseline_dev_predictions.npy: {dev_predictions_baseline.shape}")
print(f"  - ncf_baseline_test_predictions.npy: {test_predictions_baseline.shape}")
print(f"  - ncf_baseline_dev_ground_truth.npy: {dev_subreddits.shape}")
print(f"  - ncf_baseline_test_ground_truth.npy: {test_subreddits.shape}")
print(f"  - Used checkpoint: {BASELINE_CHECKPOINT_NAME}")
print("\nYou can evaluate these with:")
print("  python evaluate.py --predictions_npy ncf_baseline_test_predictions.npy --ground_truth_npy ncf_baseline_test_ground_truth.npy")

## Train NCF with Context

In [ ]:
print("="*60)
print("MODEL 2: NCF WITH CONTEXT (GMF + MLP + VW Context)")
print("="*60)
print("This model uses collaborative filtering PLUS VW text-based context.")
print("VW context (5000-dim) is projected to 128-dim and concatenated with user embeddings.")
print()

# Check if context is available
if user_context_map is None:
    print("ERROR: Context map not found!")
    print("   Run process_engage_corpus_v3.py to generate VW context first.")
    print("   Skipping context model training.")
else:
    # Create dataset and dataloader FOR CONTEXT MODEL
    train_dataset_context = NCFDatasetWithContext(train_users, train_subreddits, user_context_map,
                                                  NUM_SUBREDDITS, num_negatives=NUM_NEGATIVES, is_training=True)
    train_loader_context = DataLoader(train_dataset_context, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    
    # Create VALIDATION dataset and dataloader
    dev_dataset_context = NCFDatasetWithContext(dev_users, dev_subreddits, user_context_map,
                                               NUM_SUBREDDITS, num_negatives=NUM_NEGATIVES, is_training=True)
    dev_loader_context = DataLoader(dev_dataset_context, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Create model
    ncf_context = NCFWithContext(NUM_USERS, NUM_SUBREDDITS, 
                                embedding_dim=EMBEDDING_DIM,
                                context_dim=5000,
                                context_projection_dim=128).to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(ncf_context.parameters(), lr=LEARNING_RATE)
    
    # Check for existing checkpoint
    start_epoch = 0
    context_checkpoint_pattern = os.path.join(CHECKPOINT_DIR, 'ncf_context_epoch_*.pth')
    existing_checkpoints = sorted(glob.glob(context_checkpoint_pattern))
    if existing_checkpoints:
        latest_checkpoint = existing_checkpoints[-1]
        print(f"Loading checkpoint: {latest_checkpoint}")
        checkpoint = torch.load(latest_checkpoint)
        ncf_context.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming from epoch {start_epoch}")
    
    print(f"\nModel parameters: {sum(p.numel() for p in ncf_context.parameters()):,}")
    print(f"Training samples: {len(train_dataset_context):,}")
    print(f"Validation samples: {len(dev_dataset_context):,}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Batches per epoch: {len(train_loader_context)}")
    print()
    
    # Training loop
    for epoch in range(start_epoch, NUM_EPOCHS):
        train_loss = train_epoch_context(ncf_context, train_loader_context, criterion, optimizer, device)
        val_loss = validate_epoch_context(ncf_context, dev_loader_context, criterion, device)
        
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        # Save checkpoint every CHECKPOINT_INTERVAL epochs
        if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
            checkpoint_path = os.path.join(CHECKPOINT_DIR, f'ncf_context_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': ncf_context.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss
            }, checkpoint_path)
            print(f"  Checkpoint saved: {checkpoint_path}")
    
    # Save final model
    final_model_path = os.path.join(CHECKPOINT_DIR, 'ncf_context_final.pth')
    torch.save(ncf_context.state_dict(), final_model_path)
    print(f"\nNCF Context final model saved to {final_model_path}")

## Generate Predictions from NCF with Context

In [ ]:
print("="*60)
print("GENERATING PREDICTIONS FROM NCF WITH CONTEXT")
print("="*60)

if user_context_map is None:
    print("Skipping - context map not available")
else:
    # ============================================================
    # SPECIFY WHICH CHECKPOINT TO USE FOR PREDICTIONS
    # ============================================================
    # Options:
    # 1. Use 'ncf_context_final.pth' for the final trained model
    # 2. Use 'ncf_context_epoch_15.pth' for a specific epoch
    # 3. Use 'ncf_context_epoch_12.pth', etc.
    
    CONTEXT_CHECKPOINT_NAME = 'ncf_context_final.pth'  # Change this to use different checkpoint
    context_checkpoint_path = os.path.join(CHECKPOINT_DIR, CONTEXT_CHECKPOINT_NAME)
    
    # Verify checkpoint exists
    if not os.path.exists(context_checkpoint_path):
        print(f"ERROR: Checkpoint not found: {context_checkpoint_path}")
        print(f"\nAvailable context checkpoints:")
        context_checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'ncf_context_*.pth')))
        for ckpt in context_checkpoints:
            print(f"  - {os.path.basename(ckpt)}")
        raise FileNotFoundError(f"Checkpoint not found: {context_checkpoint_path}")
    
    print(f"Loading weights from: {CONTEXT_CHECKPOINT_NAME}")
    
    # Create a fresh model instance
    ncf_context_eval = NCFWithContext(
        NUM_USERS, 
        NUM_SUBREDDITS,
        embedding_dim=EMBEDDING_DIM,
        context_dim=5000,
        context_projection_dim=128
    ).to(device)
    
    # Load weights from checkpoint
    checkpoint_data = torch.load(context_checkpoint_path, map_location=device)
    
    # Handle both formats: full checkpoint dict or just state_dict
    if isinstance(checkpoint_data, dict) and 'model_state_dict' in checkpoint_data:
        ncf_context_eval.load_state_dict(checkpoint_data['model_state_dict'])
        print(f"Loaded checkpoint from epoch {checkpoint_data.get('epoch', 'final')}")
        if 'val_loss' in checkpoint_data:
            print(f"  Validation loss: {checkpoint_data['val_loss']:.4f}")
    else:
        # Assume it's just the state_dict
        ncf_context_eval.load_state_dict(checkpoint_data)
        print(f"Loaded model weights")
    
    ncf_context_eval.eval()
    
    # Dev predictions
    print("\nGenerating dev predictions...")
    dev_predictions_context = np.zeros((len(dev_users), NUM_SUBREDDITS), dtype=np.float32)
    for i in tqdm(range(len(dev_users)), desc="Dev predictions"):
        user_id = dev_users[i]
        context = torch.tensor(user_context_map[user_id], dtype=torch.float32)
        scores = predict_all_subreddits_context(ncf_context_eval, user_id, context, NUM_SUBREDDITS, device)
        dev_predictions_context[i] = scores
    
    # Test predictions
    print("Generating test predictions...")
    test_predictions_context = np.zeros((len(test_users), NUM_SUBREDDITS), dtype=np.float32)
    for i in tqdm(range(len(test_users)), desc="Test predictions"):
        user_id = test_users[i]
        context = torch.tensor(user_context_map[user_id], dtype=torch.float32)
        scores = predict_all_subreddits_context(ncf_context_eval, user_id, context, NUM_SUBREDDITS, device)
        test_predictions_context[i] = scores
    
    # Save predictions as numpy arrays (for evaluate.py)
    np.save('ncf_context_dev_predictions.npy', dev_predictions_context)
    np.save('ncf_context_test_predictions.npy', test_predictions_context)
    
    print("\nPredictions saved!")
    print(f"  - ncf_context_dev_predictions.npy: {dev_predictions_context.shape}")
    print(f"  - ncf_context_test_predictions.npy: {test_predictions_context.shape}")
    print(f"  - Used checkpoint: {CONTEXT_CHECKPOINT_NAME}")
    print("\nYou can evaluate these with:")
    print("  python evaluate.py --predictions_npy ncf_context_test_predictions.npy --ground_truth_npy ncf_baseline_test_ground_truth.npy")

## Quick Evaluation (In-Notebook)

In [ ]:
def quick_evaluate(predictions, ground_truth, k=10):
    """Quick evaluation of HR@K and NDCG@K."""
    n_users = len(predictions)
    hits = 0
    ndcg_sum = 0.0
    
    for i in range(n_users):
        top_k = np.argsort(predictions[i])[::-1][:k]
        
        if ground_truth[i] in top_k:
            hits += 1
            rank = np.where(top_k == ground_truth[i])[0][0] + 1
            ndcg_sum += 1.0 / np.log2(rank + 1)
    
    hr = hits / n_users
    ndcg = ndcg_sum / n_users
    
    return hr, ndcg

print("="*60)
print("IN-NOTEBOOK EVALUATION RESULTS (Quick Assessment)")
print("="*60)
print("For official results, use evaluate.py script")
print()

# Evaluate NCF Baseline
hr_base, ndcg_base = quick_evaluate(test_predictions_baseline, test_subreddits, k=10)
print("MODEL 1: NCF Baseline (No Context)")
print(f"  HR@10:   {hr_base:.4f} ({hr_base*100:.2f}%)")
print(f"  NDCG@10: {ndcg_base:.4f}")

# Evaluate NCF with Context (if available)
if user_context_map is not None:
    hr_ctx, ndcg_ctx = quick_evaluate(test_predictions_context, test_subreddits, k=10)
    print("\nMODEL 2: NCF with Context (VW + Collaborative)")
    print(f"  HR@10:   {hr_ctx:.4f} ({hr_ctx*100:.2f}%)")
    print(f"  NDCG@10: {ndcg_ctx:.4f}")
    
    # Improvement
    print("\n" + "-"*60)
    print("IMPROVEMENT FROM ADDING CONTEXT:")
    print(f"  HR@10:   +{(hr_ctx - hr_base)*100:.2f} percentage points")
    print(f"  NDCG@10: +{(ndcg_ctx - ndcg_base):.4f}")
    
    if hr_ctx > hr_base:
        print(f"  Context model performs {((hr_ctx/hr_base - 1)*100):.1f}% better!")
    else:
        print(f"  Context model underperforms baseline")

print("\n" + "="*60)
print("EXPECTED RESULTS FROM PAPER (for reference):")
print("  NCF Baseline:    HR@10 ≈ 53.5%, NDCG@10 ≈ 33.5")
print("  NCF + Context:   HR@10 ≈ 60.3%, NDCG@10 ≈ 38.5")
print("="*60)

print("\n OFFICIAL EVALUATION:")
print("For precise results using evaluate.py:")
print()
print("1. Baseline model:")
print("   python evaluate.py --predictions_npy ncf_baseline_test_predictions.npy \\")
print("                      --ground_truth_npy ncf_baseline_test_ground_truth.npy")
print()
if user_context_map is not None:
    print("2. Context model:")
    print("   python evaluate.py --predictions_npy ncf_context_test_predictions.npy \\")
    print("                      --ground_truth_npy ncf_baseline_test_ground_truth.npy")

## Download Results

In [ ]:
from google.colab import files

print("="*60)
print("DOWNLOADING RESULTS")
print("="*60)
print()

# Download final model weights
print("Downloading model weights...")
files.download(os.path.join(CHECKPOINT_DIR, 'ncf_baseline_final.pth'))
if user_context_map is not None:
    files.download(os.path.join(CHECKPOINT_DIR, 'ncf_context_final.pth'))

# Download predictions and ground truth
print("\nDownloading predictions...")
files.download('ncf_baseline_test_predictions.npy')
files.download('ncf_baseline_test_ground_truth.npy')

if user_context_map is not None:
    files.download('ncf_context_test_predictions.npy')

print("\n All files downloaded!")
print(f"\n Checkpoints are saved in Google Drive at:")
print(f"   {CHECKPOINT_DIR}")
print()
print("You can access them anytime to resume training or for inference.")
print()
print("="*60)
print("NEXT STEPS:")
print("="*60)
print()
print("1. Evaluate baseline model:")
print("   python evaluate.py --predictions_npy ncf_baseline_test_predictions.npy \\")
print("                      --ground_truth_npy ncf_baseline_test_ground_truth.npy")
print()
if user_context_map is not None:
    print("2. Evaluate context model:")
    print("   python evaluate.py --predictions_npy ncf_context_test_predictions.npy \\")
    print("                      --ground_truth_npy ncf_baseline_test_ground_truth.npy")
    print()
    print("3. Compare results to see the impact of adding VW context!")
else:
    print("2. To train the context model, run process_engage_corpus_v3.py first")
    print("   to generate VW context, then upload to Google Drive")